# Analisis Netflix Reviews: EDA, Regex, dan Bag of Words

Notebook ini berisi analisis data review Netflix. Fokus mencakup EDA, pembersihan dengan Regex, ekstraksi fitur Bag of Words, serta **Text Summarization** berbasis TF-IDF.

## 1. Import Library & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from wordcloud import WordCloud

df = pd.read_csv('../Output/Netflix_Review_Preprocessed.csv')
print(f"Dataset Shape: {df.shape}")

## 2. EDA (Exploratory Data Analysis)

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(x='score', data=df, palette='viridis')
plt.title('Distribusi Skor Review Netflix')
plt.show()

## 3. Analisis Regex & Sentiment Insight

In [ ]:
def clean_regex(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'[^a-zA-Z\s]', '', text.lower())
    return text.strip()

df['regex_cleaned'] = df['content'].apply(clean_regex)

def detect_sentiment(text):
    pos = r'\b(bagus|puas|mantap|keren|suka|love|best|good|great|enjoy|amazing)\b'
    neg = r'\b(kecewa|jelek|buruk|bad|worst|disappointed|slow|mahal|error|rugi|benci|hate)\b'
    if re.search(pos, text): return "Positive"
    if re.search(neg, text): return "Negative"
    return "Neutral"

df['sentiment_insight'] = df['regex_cleaned'].apply(detect_sentiment)
print("Sentiment Count via Regex:")
display(df['sentiment_insight'].value_counts())

## 4. Bag of Words Analysis

In [ ]:
data_text = df['final_content'].fillna('')
cv = CountVectorizer(max_features=500, stop_words=['netflix', 'the', 'app', 'to', 'and', 'is', 'it'])
bow_matrix = cv.fit_transform(data_text)

word_freq = pd.DataFrame({'Word': cv.get_feature_names_out(), 'Freq': bow_matrix.toarray().sum(axis=0)})
display(word_freq.sort_values(by='Freq', ascending=False).head(10))

## 5. Text Summarization via TF-IDF
Kita dapat merangkum kumpulan review ini dengan mencari review yang paling informatif (memiliki skor TF-IDF tertinggi).

In [ ]:
tfidf_vec = TfidfVectorizer(max_features=1000, stop_words=['netflix', 'the', 'app', 'is', 'it'])
tfidf_matrix = tfidf_vec.fit_transform(data_text)

# Skor Kepentingan Kalimat/Review = rata-rata skor TF-IDF
review_scores = np.asarray(tfidf_matrix.mean(axis=1)).ravel()
df_summary = pd.DataFrame({'Original Review': df['content'], 'Score': review_scores}).sort_values(by='Score', ascending=False)

print("TOP 5 MOST INFORMATIVE REVIEWS (SUMMARIZATION):")
display(df_summary.head(5))

## 6. WordCloud Visualization

In [ ]:
all_words = ' '.join(data_text[:5000]) # Sample for speed
wc = WordCloud(width=800, height=400, background_color='white').generate(all_text)
plt.figure(figsize=(12, 6))
plt.imshow(wc)
plt.axis('off')
plt.show()